# Web Scraping Workshop

## Proactieve Zorgbemiddeling: hoe lang wacht Nederland op zorg?

In deze workshop leer je een echte website scrapen. We gebruiken **Zorgkaart Nederland** als bron, vergelijken werkelijke wachttijden met de wettelijke normen, en bouwen tot slot een interactieve **Wachttijden Radar** waarmee je per specialisme kunt zien welke ziekenhuizen te lang laten wachten.

Onderweg schrijf je zelf delen van de scraper: dit is een doe-workshop, geen demo.

In [1]:
# Het installeren van de benodigde pakketten

%pip install requests beautifulsoup4 lxml pandas plotly

Note: you may need to restart the kernel to use updated packages.


## Achtergrond

### Proactieve zorgbemiddeling
Sinds 2026 is je zorgverzekeraar wettelijk verplicht je *actief* te helpen als je te lang op zorg moet wachten. Niet meer "bel maar als je wilt", maar de verzekeraar moet zélf op zoek naar een ziekenhuis met kortere wachttijd. Dat heet **proactieve zorgbemiddeling**.

### Treeknormen
Hoe lang is "te lang"? Dat staat in de **treeknormen** — afspraken tussen de Nederlandse Zorgautoriteit (NZa) en de zorgverzekeraars over maximale wachttijden:
- **Polikliniekbezoek**: max 4 weken (28 dagen)
- **Diagnostisch onderzoek**: max 4 weken (28 dagen)
- **Behandeling**: max 7 weken (49 dagen)

Overschrijdt een ziekenhuis deze norm? Dan moet je verzekeraar je proactief elders plaatsen.

### Onze databron: Zorgkaart Nederland
We gebruiken **[Zorgkaart Nederland](https://www.zorgkaartnederland.nl)**: een platform van patiëntenfederatie Nederland waar ziekenhuizen zelf hun actuele wachttijden publiceren. Per specialisme staat er een tabel met alle aanbieders en hun wachttijd in dagen — precies wat we nodig hebben.

### Wat gaan we doen?
1. Een wachttijdenpagina ophalen met `requests`
2. HTML parsen met `BeautifulSoup`
3. Wachttijden extraheren uit de tabel
4. De logica in een functie stoppen
5. Alle specialismen scrapen
6. Vergelijken met de treeknormen
7. Visualiseren
8. Een interactieve **Wachttijden Radar** bouwen

## Achtergrond: Compliance

Webscraping is an sich legaal, maar soms moet je oppassen hoe je het doet en wat je met de gescrapde data doet.<br>
<br>
1. Je bent nog altijd behouden aan wetgeving omtrent copyright. Je mag dus niet creatieve content scrapen en herpubliceren. <br>
Maar 'feit'-data (e.g. wachttijden, weer, financiële koersen, etc.) valt daar niet onder.
2. De Terms of Service (van de website) is een contract tussen jou en de website-beheerder over hoe je dient om te gaan met de website en de inhoud. <br>
Als scrapen daar wordt verboden, neem je dus een risico op contractbreuk.

Robot Exclusion Protocol is opgenomen in Robots.txt. Dit is een tekstbestand dat vaak in de thuis-folder van een website geplaatst wordt. <br>
Dit dient als informatie voor de geautomatiseerde bots waarin staat welke delen van de website zij mogen benaderen.<br>
<br>Vaak omvat het Robots.txt bestand informatie over <br>
1. Welke bot de regels betreffen (e.g. een specifieke bot bijv. van Google of OpenAI, of alle bots)
2. Welke URLs de bot niet mag scrapen of welke URLs de bot enkel mag scrapen
3. Een crawl-delay: aantal seconden tussen verzoeken de bot moet wachten om de server niet te crashen.<br><br>

Best practices:
1. Respecteer Robots.txt
2. Gebruik een crawl-delay om niet duizenden verzoeken per seconden te sturen (kan geclassificeerd worden als een DoS aanval)
3. Gebruik identificeerbare headers met je contact-informatie
4. Scrape alleen wat je nodig hebt
5. Check of er geen officiële API beschikbaar is om ontwikkelaars toegang te geven tot data.

Zie de robots.txt van zorgkaartnederland:

https://www.zorgkaartnederland.nl/robots.txt

Je ziet informatie over het verbod op scrapen om AI-modellen te trainen, specifieke andere bots, maar daar buiten <br>
zie je toestemming voor meeste andere bots, zolang je niet er een AI mee traint.

## Stap 1: Een wachttijdenpagina ophalen

Een webpagina ophalen werkt zo: jouw computer stuurt een **request** naar de server, en die stuurt een **response** terug. De response bevat een statuscode (kreeg ik wat ik vroeg?) en de inhoud van de pagina als ruwe HTML.

**Statuscodes in een notendop:**
- `200` — OK, alles goed
- `403` — verboden (server wil je niet helpen)
- `404` — pagina bestaat niet
- `429` — te veel requests, even rustig aan

We halen de cardiologie-pagina op.

In [2]:
import requests

url = "https://www.zorgkaartnederland.nl/wachttijden/cardiologie-2"

# Een User-Agent vertelt de server wie er aanklopt. Sommige sites blokkeren
# het standaard Python-user-agent, dus doen we ons voor als een gewone browser.
headers = {"User-Agent": "Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36"}

response = requests.get(url, headers=headers)

print(f"Status code: {response.status_code}")  # 200 = succes
print(f"Grootte: {len(response.text):,} tekens")

# response.text is de ruwe HTML als één lange string
print(f"\nEerste 500 tekens:\n{response.text[:500]}")

Status code: 200
Grootte: 197,106 tekens

Eerste 500 tekens:
<!DOCTYPE html>
<html lang="nl" class="no-js" xmlns="http://www.w3.org/1999/xhtml">
<head>
    <meta charset="utf-8">
    <meta name="viewport" content="width=device-width, initial-scale=1.0, minimum-scale=1.0, maximum-scale=1.0, user-scalable=no" />
    <meta content="IE=edge" http-equiv="X-UA-Compatible">
    <meta name="format-detection" content="telephone=no">
    <title>            Wachttijden voor polikliniek cardiologie
    </title>

    <link rel="canonical" href="https://www.zorgkaartne


## Stap 2: HTML parsen met BeautifulSoup

De ruwe HTML uit stap 1 is één lange string, maar HTML is eigenlijk een **boomstructuur**: tags binnen tags binnen tags. Met string-zoeken kom je er niet — je hebt een parser nodig die die boom voor je opbouwt. Daar is **BeautifulSoup** voor.

Eenmaal geparsed kun je in de boom zoeken met **CSS-selectors**, dezelfde syntax die je in stylesheets gebruikt:
- `tag` — alle elementen van dat type (bv. `tr` voor alle tabelrijen)
- `.klassenaam` — alle elementen met die CSS-class
- `tag.klassenaam` — combinatie (bv. `table.table-certificates`)

Twee methodes om te zoeken:
- `select_one(...)` — pakt het **eerste** element dat matcht
- `select(...)` — pakt **alle** elementen die matchen (geeft een lijst)

In [3]:
from bs4 import BeautifulSoup

# lxml is een snelle, robuuste HTML-parser
soup = BeautifulSoup(response.text, "lxml")

# de tabel met wachttijden heeft de class "table-certificates"
table = soup.select_one("table.table-certificates")
print(f"Tabel gevonden: {table is not None}")

# tr = "table row" — één rij in de tabel
rows = table.select("tr")
print(f"Aantal rijen: {len(rows)} (1 header + {len(rows)-1} ziekenhuizen)")

# th = "table header" — de kolomkoppen staan in de eerste rij
header = [th.get_text(strip=True) for th in rows[0].select("th")]
print(f"Kolommen: {header}")

print("\n--- Eerste 3 rijen ---")
for row in rows[1:4]:
    # td = "table data" — een gewone cel in een tabelrij
    cells = [td.get_text(strip=True) for td in row.select("td")]
    print(cells)

Tabel gevonden: True
Aantal rijen: 183 (1 header + 182 ziekenhuizen)
Kolommen: ['Zorgaanbieder', 'Locatie', 'Wachttijdin dagen']

--- Eerste 3 rijen ---
['Cardiologiecentrum Care for Heart', 'Zoetermeer', '1']
['Ksyos, het digitale ziekenhuis', 'Amsterdam', '1']
['Stichting Cardiologie Amsterdam', 'Amsterdam', '2']


## Stap 3: Eén rij ontleden — **jij aan de beurt**

We weten nu hoe we cellen uit een rij halen. Tijd om er bruikbare velden van te maken: naam, locatie en wachttijd. Vul de drie variabelen hieronder zelf in.

**Tips:**
- Een datarij heeft drie cellen: `cells[0]`, `cells[1]`, `cells[2]`.
- `.get_text(strip=True)` haalt platte tekst eruit, zonder witregels of spaties eromheen.
- Wachttijd staat als tekst (`"35"`) — gebruik `int(...)` om er een geheel getal van te maken.

In [4]:
row = rows[1]               # eerste data-rij (rows[0] was de header)
cells = row.select("td")    # de drie cellen van die rij

# Antwoord: cellen op index 0/1/2, telkens met .get_text(strip=True);
# voor wachttijd zetten we de tekst om naar int.
naam = cells[0].get_text(strip=True)
locatie = cells[1].get_text(strip=True)
wachttijd_dagen = int(cells[2].get_text(strip=True))

print(f"Naam: {naam}")
print(f"Locatie: {locatie}")
print(f"Wachttijd: {wachttijd_dagen} dagen")

Naam: Cardiologiecentrum Care for Heart
Locatie: Zoetermeer
Wachttijd: 1 dagen


In [5]:
# Check: dit cel-tje controleert of je velden van het juiste type zijn.
# Als deze cel zonder fout draait, zit je goed.
assert isinstance(naam, str) and len(naam) > 0, "naam moet een niet-lege string zijn"
assert isinstance(locatie, str), "locatie moet een string zijn"
assert isinstance(wachttijd_dagen, int), "wachttijd_dagen moet een int zijn (gebruik int(...))"
print("Top, alle velden hebben het juiste type.")

Top, alle velden hebben het juiste type.


## Stap 4: Functie voor één specialisme — **jij aan de beurt**

In stap 3 had je drie regels voor één rij. Voor 50 ziekenhuizen wil je dat niet kopiëren — en zeker niet als we straks 12 specialismen aflopen. We stoppen de logica daarom in een **functie**.

Eén principe vooraf: **defensief programmeren**. Een echte website is rommelig — soms ontbreekt de tabel, soms is een rij stuk, soms staat er "Onbekend" in plaats van een getal. Je code moet daar tegen kunnen zonder vast te lopen.

De functieschets staat hieronder. **Vul de twee ontbrekende velden** in het dict zelf in.

In [6]:
# Door de logica in een functie te zetten, kunnen we 'm later voor elke
# specialisme-URL hergebruiken (zie stap 5).
def scrape_wachttijden(url):
    """Scrape alle wachttijden van één Zorgkaart Nederland-pagina."""
    response = requests.get(url, headers=headers)
    soup = BeautifulSoup(response.text, "lxml")

    table = soup.select_one("table.table-certificates")
    if not table:
        return []  # geen tabel gevonden? dan stoppen we netjes met een lege lijst

    resultaten = []
    for row in table.select("tr")[1:]:  # [1:] slaat de header-rij over
        cells = row.select("td")
        if len(cells) < 3:
            continue  # onvolledige/rare rij — overslaan

        # wachttijd kan ook "Onbekend" zijn; alleen omzetten als het echt een getal is
        wachttijd_tekst = cells[2].get_text(strip=True)
        wachttijd_dagen = int(wachttijd_tekst) if wachttijd_tekst.isdigit() else None

        # de hyperlink naar het ziekenhuisprofiel zit in een <a>-tag in cells[0];
        # via ["href"] pakken we het link-pad en plakken de basis-URL ervoor
        link_el = cells[0].select_one("a")
        profiel_url = "https://www.zorgkaartnederland.nl" + link_el["href"] if link_el else ""

        resultaten.append({
            "zorgaanbieder": cells[0].get_text(strip=True),
            "locatie": cells[1].get_text(strip=True),
            "wachttijd_dagen": wachttijd_dagen,
            "profiel_url": profiel_url,
        })

    return resultaten


resultaten = scrape_wachttijden("https://www.zorgkaartnederland.nl/wachttijden/cardiologie-2")
print(f"Gevonden: {len(resultaten)} ziekenhuizen")
resultaten[:3]

Gevonden: 182 ziekenhuizen


[{'zorgaanbieder': 'Cardiologiecentrum Care for Heart',
  'locatie': 'Zoetermeer',
  'wachttijd_dagen': 1,
  'profiel_url': 'https://www.zorgkaartnederland.nl/zorginstelling/cardiologiecentrum-cardiologiecentrum-care-for-heart-zoetermeer-3025143/wachttijden#wachttijd-cardiologie-2'},
 {'zorgaanbieder': 'Ksyos, het digitale ziekenhuis',
  'locatie': 'Amsterdam',
  'wachttijd_dagen': 1,
  'profiel_url': 'https://www.zorgkaartnederland.nl/zorginstelling/overige-kliniek-ksyos-het-digitale-ziekenhuis-amsterdam-10010380/wachttijden#wachttijd-cardiologie-2'},
 {'zorgaanbieder': 'Stichting Cardiologie Amsterdam',
  'locatie': 'Amsterdam',
  'wachttijd_dagen': 2,
  'profiel_url': 'https://www.zorgkaartnederland.nl/zorginstelling/cardiologiecentrum-stichting-cardiologie-amsterdam-amsterdam-3049112/wachttijden#wachttijd-cardiologie-2'}]

In [7]:
# Check: hebben we een lijst dicts met de juiste keys?
assert isinstance(resultaten, list) and len(resultaten) > 0, "resultaten moet een niet-lege lijst zijn"
eerste = resultaten[0]
assert set(eerste.keys()) == {"zorgaanbieder", "locatie", "wachttijd_dagen", "profiel_url"}, \
    "elke entry moet vier keys hebben"
assert isinstance(eerste["zorgaanbieder"], str) and len(eerste["zorgaanbieder"]) > 0, \
    "zorgaanbieder moet een niet-lege string zijn"
assert isinstance(eerste["locatie"], str), "locatie moet een string zijn"
print(f"Top, je hebt {len(resultaten)} ziekenhuizen correct gescraped.")

Top, je hebt 182 ziekenhuizen correct gescraped.


## Stap 5: Meerdere specialismen scrapen — **jij aan de beurt**

Onze functie werkt voor één URL. Met een loop over een dict scrapen we ze allemaal in één keer.

**URL-patroon:** Zorgkaart Nederland gebruikt steeds dezelfde structuur:
`https://www.zorgkaartnederland.nl/wachttijden/{slug}` — waarbij `{slug}` iets is als `cardiologie-2`, `dermatologie` of `mri-hersenen`.

**Etiquette:** tussen requests pauzeren we even (`time.sleep(0.5)`). Webservers vinden 12 requests in 50 ms niet leuk en kunnen je tijdelijk blokkeren. Niet sneller scrapen dan nodig.

**Jouw beurt:** voeg minstens **één eigen specialisme** toe aan de dict hieronder. Ga naar [zorgkaartnederland.nl/wachttijden](https://www.zorgkaartnederland.nl/wachttijden), kies er een en haal de slug uit de URL.

In [8]:
import time
import pandas as pd

# Sleutel = de slug uit de URL. Waarde = (leesbare naam, type voor de treeknorm).
# Het 'type' moet "polikliniek", "diagnostiek" of "behandeling" zijn.
specialismen = {
    "cardiologie-2": ("Cardiologie", "polikliniek"),
    "chirurgie-heelkunde": ("Chirurgie", "polikliniek"),
    "dermatologie": ("Dermatologie", "polikliniek"),
    "neurologie-2": ("Neurologie", "polikliniek"),
    "oogheelkunde-2": ("Oogheelkunde", "polikliniek"),
    "orthopedie-2": ("Orthopedie", "polikliniek"),
    "ct": ("CT-scan", "diagnostiek"),
    "echografie-radiologie": ("Echografie", "diagnostiek"),
    "mri-hersenen": ("MRI hersenen", "diagnostiek"),
    "initiele-staaroperatie-oogheelkunde": ("Staaroperatie", "behandeling"),
    "initiele-totale-knie-vervanging-orthopedie": ("Knievervanging", "behandeling"),
    "initiele-totale-heupvervanging-orthopedie": ("Heupvervanging", "behandeling"),
    # Eigen specialisme — gevonden via zorgkaartnederland.nl/wachttijden:
    "longgeneeskunde": ("Longgeneeskunde", "polikliniek"),
}

alle_data = []
for slug, (naam, wtype) in specialismen.items():
    url = f"https://www.zorgkaartnederland.nl/wachttijden/{slug}"
    resultaten = scrape_wachttijden(url)
    for r in resultaten:
        r["specialisme"] = naam
        r["type"] = wtype
    alle_data.extend(resultaten)
    print(f"  {naam}: {len(resultaten)} ziekenhuizen")
    time.sleep(0.5)  # scraping-etiquette: even pauzeren tussen requests

# Een list-of-dicts is in pandas direct om te zetten naar een DataFrame
df = pd.DataFrame(alle_data)
print(f"\nTotaal: {len(df)} rijen, {df['specialisme'].nunique()} specialismen")
df.head(10)

  Cardiologie: 182 ziekenhuizen


  Chirurgie: 167 ziekenhuizen


  Dermatologie: 214 ziekenhuizen


  Neurologie: 179 ziekenhuizen


  Oogheelkunde: 173 ziekenhuizen


...

  Knievervanging: 116 ziekenhuizen


  Heupvervanging: 115 ziekenhuizen


  Longgeneeskunde: 128 ziekenhuizen



Totaal: 1934 rijen, 13 specialismen


,zorgaanbieder,locatie,wachttijd_dagen,profiel_url,specialisme,type
0,Cardiologiecentrum Care for Heart,Zoetermeer,1.0,https://www.zorgkaartnederland.nl/zorginstelli...,Cardiologie,polikliniek
1,"Ksyos, het digitale ziekenhuis",Amsterdam,1.0,https://www.zorgkaartnederland.nl/zorginstelli...,Cardiologie,polikliniek
2,Stichting Cardiologie Amsterdam,Amsterdam,2.0,https://www.zorgkaartnederland.nl/zorginstelli...,Cardiologie,polikliniek
3,Cardiologie Centrum Almere,Almere,3.0,https://www.zorgkaartnederland.nl/zorginstelli...,Cardiologie,polikliniek
4,Cardiologie Centrum Amsterdam Slotervaart,Amsterdam,3.0,https://www.zorgkaartnederland.nl/zorginstelli...,Cardiologie,polikliniek
5,PoliDirect Tilburg,Tilburg,3.0,https://www.zorgkaartnederland.nl/zorginstelli...,Cardiologie,polikliniek
6,St. Antonius Cardicare,Blaricum,3.0,https://www.zorgkaartnederland.nl/zorginstelli...,Cardiologie,polikliniek
7,Cardiologie Centrum Amsterdam Zuid,Amsterdam,4.0,https://www.zorgkaartnederland.nl/zorginstelli...,Cardiologie,polikliniek
8,Cardiologie Centrum Hoogvliet,Hoogvliet,4.0,https://www.zorgkaartnederland.nl/zorginstelli...,Cardiologie,polikliniek
9,ETZ Elisabeth,Tilburg,4.0,https://www.zorgkaartnederland.nl/zorginstelli...,Cardiologie,polikliniek


In [9]:
# Check: heb je je eigen specialisme toegevoegd?
assert len(specialismen) > 12, "Voeg minstens één eigen specialisme toe aan de dict hierboven!"
assert df["specialisme"].nunique() >= 13, "Je eigen specialisme zit niet in de DataFrame — klopt de slug?"
eigen = df["specialisme"].nunique() - 12
print(f"Top, je hebt {eigen} eigen specialisme(s) toegevoegd, totaal {len(df)} rijen.")

Top, je hebt 1 eigen specialisme(s) toegevoegd, totaal 1934 rijen.


## Stap 6: Treeknorm-analyse

Tijd om de gescrapete data tegen de wettelijke normen te leggen. Per ziekenhuis bepalen we: zit de wachttijd onder of boven de treeknorm? En in welk type zorg is het probleem het grootst?

Belangrijk: een overschrijding is geen statistiek maar concrete realiteit — voor al die patiënten moet de verzekeraar proactief gaan bemiddelen.

In [10]:
# Treeknormen: maximale wachttijden per type zorg, in dagen.
treeknormen = {
    "polikliniek": 28,   # 4 weken
    "diagnostiek": 28,   # 4 weken
    "behandeling": 49,   # 7 weken
}

# .map() vervangt elke waarde in df["type"] door de bijbehorende waarde uit de dict
df["treeknorm_dagen"] = df["type"].map(treeknormen)

# een vergelijking tussen twee kolommen levert een boolean-kolom op (True/False per rij)
df["overschrijding"] = df["wachttijd_dagen"] > df["treeknorm_dagen"]

# .sum() op een boolean-kolom telt het aantal True's
n_totaal = len(df.dropna(subset=["wachttijd_dagen"]))
n_overschrijding = df["overschrijding"].sum()
pct = n_overschrijding / n_totaal * 100

print(f"{n_overschrijding} van {n_totaal} ziekenhuizen ({pct:.0f}%) overschrijdt de treeknorm.")
print("Voor al deze ziekenhuizen geldt: hun verzekeraar moet bemiddeling aanbieden.\n")

# groupby per specialisme; mean() op boolean = fractie True's = % overschrijding
print("Percentage overschrijding per specialisme:")
print(df.groupby("specialisme")["overschrijding"].mean().mul(100).round(0).sort_values(ascending=False).to_string())

880 van 1856 ziekenhuizen (47%) overschrijdt de treeknorm.
Voor al deze ziekenhuizen geldt: hun verzekeraar moet bemiddeling aanbieden.

Percentage overschrijding per specialisme:
specialisme
Neurologie         79.0
Oogheelkunde       64.0
Longgeneeskunde    61.0
Dermatologie       59.0
MRI hersenen       50.0
Cardiologie        48.0
Knievervanging     48.0
Heupvervanging     47.0
CT-scan            33.0
Staaroperatie      33.0
Orthopedie         23.0
Chirurgie          19.0
Echografie         15.0


### Eigen analyse — **jij aan de beurt**

Je hebt nu een DataFrame met alle ziekenhuizen, hun wachttijd, type en of ze de norm overschrijden. Kies **één eigen vraag** en beantwoord 'm in de cel hieronder. Een paar suggesties:

- Welke individuele zorgaanbieder heeft gemiddeld de langste wachttijden? (groeperen op `zorgaanbieder`)
- In welk specialisme is de spreiding het grootst (verschil tussen kortste en langste wachttijd)?
- Bij welke locatie/stad valt het op dat ziekenhuizen consequent over de norm zitten?

Tip: `df.groupby("kolom")["wachttijd_dagen"].mean()` of `.agg([min, max])` werkt vaak prima.

In [11]:
# TODO: schrijf hier je eigen analyse
# Voorbeeld om mee te beginnen (voel je vrij dit te overschrijven):
df.groupby("zorgaanbieder")["wachttijd_dagen"].mean().sort_values(ascending=False).head(10)

zorgaanbieder
Stichting CardioZorg                                              212.000000
Isala Kampen                                                      203.500000
MST-Eyescan Enschede                                              165.000000
SEIN, Polikliniek voor epilepsie, locatie Cruquius                145.000000
Oogkliniek Parkstad, een focuskliniek van Bergman Clinics         125.000000
Oogkliniek Noord-Limburg, een focuskliniek van Bergman Clinics    125.000000
Oogkliniek Limburg, een focuskliniek van Bergman Clinics          125.000000
SEIN, Slaap-Waakcentrum, locatie Groningen                        124.000000
Isala Heerde                                                      123.000000
Isala Meppel                                                      121.909091
Name: wachttijd_dagen, dtype: float64

In [12]:
print("De 15 langste wachttijden in onze dataset:\n")
top15 = df.nlargest(15, "wachttijd_dagen")[["specialisme", "zorgaanbieder", "locatie", "wachttijd_dagen", "treeknorm_dagen"]].copy()
top15["overschrijding_dagen"] = top15["wachttijd_dagen"] - top15["treeknorm_dagen"]
print(top15.to_string(index=False))

De 15 langste wachttijden in onze dataset:

  specialisme                                                            zorgaanbieder     locatie  wachttijd_dagen  treeknorm_dagen  overschrijding_dagen
 Oogheelkunde                                                             Isala Kampen      Kampen            884.0               28                 856.0
 Oogheelkunde                                                             Isala Meppel      Meppel            884.0               28                 856.0
 Oogheelkunde                                                             Isala Zwolle      Zwolle            884.0               28                 856.0
   Neurologie                                          Ommelander Ziekenhuis Groningen    Scheemda            500.0               28                 472.0
Staaroperatie                                                       Martini Ziekenhuis   Groningen            402.0               49                 353.0
   Neurologie             

## Stap 7: Visualiseren

Een goede visualisatie begint niet bij het chart-type maar bij de vraag. Twee veelvoorkomende:

- **Vergelijken** (welk specialisme zit het slechtst?) → een **bar chart**: alle waarden naast elkaar.
- **Spreiding** (hoeveel variatie zit er tussen ziekenhuizen?) → een **box plot**: min, max, mediaan en uitschieters in één beeld.

We bouwen elke viz in **twee stappen** op: eerst de aggregatie als losse DataFrame (zodat je ziet wat je gaat plotten), daarna pas de grafiek.

### Percentage overschrijding per specialisme

In [13]:
# Stap 1 — aggregeren: percentage overschrijding per specialisme
pct_per_spec = (
    df.groupby(["specialisme", "type"])["overschrijding"]
    .mean()
    .mul(100)
    .round(1)
    .reset_index()
    .rename(columns={"overschrijding": "pct_overschrijding"})
    .sort_values("pct_overschrijding", ascending=True)
)
pct_per_spec

,specialisme,type,pct_overschrijding
4,Echografie,diagnostiek,14.7
2,Chirurgie,polikliniek,19.2
11,Orthopedie,polikliniek,23.0
0,CT-scan,diagnostiek,32.7
12,Staaroperatie,behandeling,33.3
5,Heupvervanging,behandeling,47.0
1,Cardiologie,polikliniek,47.8
6,Knievervanging,behandeling,48.3
8,MRI hersenen,diagnostiek,50.5
3,Dermatologie,polikliniek,58.9


In [14]:
import plotly.express as px

# Stap 2 — plotten: pas als de aggregatie klopt, gaan we tekenen
fig = px.bar(
    pct_per_spec,
    x="pct_overschrijding",
    y="specialisme",
    color="type",
    orientation="h",
    title="Percentage ziekenhuizen dat de treeknorm overschrijdt",
    labels={"pct_overschrijding": "% overschrijding", "specialisme": "", "type": "Type"},
    color_discrete_map={"polikliniek": "#2196F3", "diagnostiek": "#FF9800", "behandeling": "#9C27B0"},
)
fig.update_layout(height=500)
fig.show()

### Gemiddelde wachttijd per specialisme

Zelfde aanpak: eerst de aggregatie als DataFrame, dan plotten.

In [15]:
# Stap 1 — aggregeren: gemiddelde wachttijd per specialisme
gem = (
    df.groupby(["specialisme", "type"])
    .agg(gem_wachttijd=("wachttijd_dagen", "mean"), treeknorm=("treeknorm_dagen", "first"))
    .reset_index()
    .sort_values("gem_wachttijd", ascending=True)
)
gem

,specialisme,type,gem_wachttijd,treeknorm
2,Chirurgie,polikliniek,17.578616,28
4,Echografie,diagnostiek,18.259542,28
11,Orthopedie,polikliniek,19.710059,28
0,CT-scan,diagnostiek,25.463918,28
8,MRI hersenen,diagnostiek,28.731481,28
1,Cardiologie,polikliniek,32.219101,28
7,Longgeneeskunde,polikliniek,41.070866,28
3,Dermatologie,polikliniek,41.302439,28
12,Staaroperatie,behandeling,45.578125,49
5,Heupvervanging,behandeling,60.136364,49


In [16]:
# Stap 2 — plotten met de aggregatie als input
fig = px.bar(
    gem,
    x="gem_wachttijd",
    y="specialisme",
    orientation="h",
    title="Gemiddelde wachttijd per specialisme (in dagen)",
    labels={"gem_wachttijd": "Gemiddelde wachttijd (dagen)", "specialisme": ""},
    color="type",
    color_discrete_map={"polikliniek": "#2196F3", "diagnostiek": "#FF9800", "behandeling": "#9C27B0"},
)

# verticale streep op 28 dagen = treeknorm voor poli/diagnostiek
fig.add_vline(x=28, line_dash="dash", line_color="#F44336", annotation_text="Treeknorm poli/diag (28d)")
fig.update_layout(height=500)
fig.show()

### Spreiding per specialisme — **jij aan de beurt**

Voor de derde viz is geen aggregatie nodig: een box plot werkt direct op de ruwe `df`. Plotly express heeft daar `px.box(...)` voor — qua aanroep heel vergelijkbaar met `px.bar`. Vul het skelet hieronder zelf in.

**Hint:** voor een horizontale box plot wil je `wachttijd_dagen` op de **x**-as, `specialisme` op de **y**-as, kleur op `"type"`, en `orientation="h"`.

In [17]:
fig = px.box(
    df,
    x="wachttijd_dagen",
    y="specialisme",
    color="type",
    orientation="h",
    title="Spreiding wachttijden per specialisme",
    labels={"wachttijd_dagen": "Wachttijd (dagen)", "specialisme": ""},
    color_discrete_map={"polikliniek": "#2196F3", "diagnostiek": "#FF9800", "behandeling": "#9C27B0"},
)
# stippellijntjes voor de twee treeknormen
fig.add_vline(x=28, line_dash="dash", line_color="#F44336", opacity=0.5)
fig.add_vline(x=49, line_dash="dash", line_color="#F44336", opacity=0.5)
fig.update_layout(height=600)
fig.show()

## Stap 8: Interactieve Wachttijden Radar

Tijd voor het eindproduct. We gebruiken hier `plotly.graph_objects` (een laag onder `plotly.express`) omdat we precieze controle willen over wat zichtbaar is. Het idee:

- Voor elk specialisme maken we een aparte **trace** (één set bars).
- Per default zetten we alleen de eerste op `visible=True`.
- Een **dropdown** met buttons toggelt zichtbaarheid: kies een specialisme, en alleen die trace wordt getoond.

Zo bouw je interactiviteit zonder framework — het is een figuur met meerdere lagen waarvan de gebruiker via knoppen kiest welke laag zichtbaar is.

In [18]:
import plotly.graph_objects as go

specialisme_namen = sorted(df["specialisme"].unique())
fig = go.Figure()

for i, spec in enumerate(specialisme_namen):
    subset = df[df["specialisme"] == spec].dropna(subset=["wachttijd_dagen"]).sort_values("wachttijd_dagen")
    tk = subset["treeknorm_dagen"].iloc[0]
    kleuren = ["#4CAF50" if w <= tk else "#F44336" for w in subset["wachttijd_dagen"]]
    labels = [f"{r.zorgaanbieder} ({r.locatie})" for _, r in subset.iterrows()]
    fig.add_trace(go.Bar(x=subset["wachttijd_dagen"].values, y=labels, orientation="h",
                         marker_color=kleuren, visible=(i == 0), name=spec))

buttons = []
for i, spec in enumerate(specialisme_namen):
    sub = df[df["specialisme"] == spec]
    tk = sub["treeknorm_dagen"].iloc[0]
    n_over = (sub["wachttijd_dagen"] > tk).sum()
    vis = [False] * len(specialisme_namen)
    vis[i] = True
    buttons.append(dict(label=spec, method="update", args=[
        {"visible": vis},
        {"title": f"Wachttijden Radar: {spec}<br><sub>{n_over}/{len(sub)} overschrijdt treeknorm ({tk}d)</sub>",
         "shapes": [dict(type="line", x0=tk, x1=tk, y0=-0.5, y1=len(sub)-0.5,
                         line=dict(color="#FF9800", width=2, dash="dash"))]}]))

first = df[df["specialisme"] == specialisme_namen[0]]
ftk = first["treeknorm_dagen"].iloc[0]
fig.update_layout(
    title=f"Wachttijden Radar: {specialisme_namen[0]}<br><sub>{(first['wachttijd_dagen']>ftk).sum()}/{len(first)} overschrijdt treeknorm ({ftk}d)</sub>",
    updatemenus=[dict(buttons=buttons, direction="down", showactive=True, x=0, xanchor="left", y=1.15, yanchor="top")],
    xaxis_title="Wachttijd (dagen)", height=max(600, len(first)*18), margin=dict(l=300),
    shapes=[dict(type="line", x0=ftk, x1=ftk, y0=-0.5, y1=len(first)-0.5, line=dict(color="#FF9800", width=2, dash="dash"))])
fig.show()

In [19]:
fig.write_html("wachttijden_radar.html", include_plotlyjs=True)
print("Opgeslagen: wachttijden_radar.html")
print("Open dit bestand in je browser voor de interactieve Wachttijden Radar!")

df.to_csv("wachttijden_zorg.csv", index=False)
print(f"\nData opgeslagen: wachttijden_zorg.csv ({len(df)} rijen)")

Opgeslagen: wachttijden_radar.html
Open dit bestand in je browser voor de interactieve Wachttijden Radar!

Data opgeslagen: wachttijden_zorg.csv (1934 rijen)


## Bonus: detailpagina's volgen

Klaar met de hoofdoefeningen? Tijd voor één laagje dieper.

Elk ziekenhuis in onze dataset heeft een `profiel_url`. Volg je die, dan kom je op de detailpagina — daar staan dingen als de **rating** (gemiddelde score uit 5) en het **aantal beoordelingen**. Die zou je per ziekenhuis kunnen ophalen om de analyse te verrijken.

**De DevTools-truc:** open zo'n profielpagina in je browser, klik rechts op het element dat je wilt scrapen (bv. de rating), kies *Inspecteer* / *Inspect element*. Je ziet de HTML rond dat element — daaruit haal je de juiste CSS-selector. Zoek de class waar de rating in staat, en gebruik die met `soup.select_one(...)`.

Probeer hieronder voor één ziekenhuis de rating en het aantal beoordelingen te scrapen.

In [20]:
# Pak één ziekenhuis uit de dataset om mee te oefenen
voorbeeld = df.iloc[0]
print(f"We scrapen: {voorbeeld['zorgaanbieder']}")
print(f"URL: {voorbeeld['profiel_url']}\n")

# Het 'profiel_url' wijst naar de wachttijden-tab; we willen de hoofdpagina,
# dus halen we het '/wachttijden' suffix eraf.
import re
profiel_root = re.sub(r"/wachttijden$", "", voorbeeld["profiel_url"])

response = requests.get(profiel_root, headers=headers)
soup = BeautifulSoup(response.text, "lxml")

# De rating staat in een <div class="ratinge__score">. Het integerdeel is de
# directe tekst, het decimaaldeel zit in een geneste <div class="ratinge__score-decimal">.
# get_text(strip=True) plakt beide netjes aan elkaar tot bv. "8.9".
rating_el = soup.select_one("div.ratinge__score")
rating = rating_el.get_text(strip=True) if rating_el else None

# Het aantal waarderingen staat in een link in p[data-sticky-header="score-text"],
# bv. "38 waarderingen".
aantal_el = soup.select_one('p[data-sticky-header="score-text"] a')
aantal_beoordelingen = aantal_el.get_text(strip=True) if aantal_el else None

print(f"Rating: {rating}")
print(f"Aantal beoordelingen: {aantal_beoordelingen}")

We scrapen: Cardiologiecentrum Care for Heart
URL: https://www.zorgkaartnederland.nl/zorginstelling/cardiologiecentrum-cardiologiecentrum-care-for-heart-zoetermeer-3025143/wachttijden#wachttijd-cardiologie-2

Rating: 10.0
Aantal beoordelingen: 2 waarderingen


## Recap

- **Webpagina ophalen**: `requests.get(url)`
- **HTML parsen**: `BeautifulSoup(html, "lxml")`
- **Tabel zoeken**: `soup.select_one("table.table-certificates")`
- **Rijen doorlopen**: `table.select("tr")[1:]`
- **Meerdere pagina's**: Loop + `time.sleep()`
- **Data structureren**: `pd.DataFrame(lijst_van_dicts)`
- **Interactieve viz**: `plotly` + `fig.write_html()`

### Zelf verder?
- Meer specialismen: [/wachttijden/poliklinieken](https://www.zorgkaartnederland.nl/wachttijden/poliklinieken)
- De overzichtspagina scrapen om alle links automatisch te vinden
- Per ziekenhuis de detailpagina scrapen